# 🥞 10.17 Union & Deduplication

Welcome to the second part of **SECTION E: COMBINING DATA**!

In the last lesson (**10.16 Joins in Spark**), we learned how to combine datasets *horizontally*. We put the `Employees` table next to the `Departments` table, adding new columns to our data.

But what if you want to combine datasets *vertically*? 
Imagine you have a file containing "January Sales" and another file containing "February Sales". You don't want to join them side-by-side; you want to stack them on top of each other to create one massive "Yearly Sales" table.

In this lesson, we will learn how to stack DataFrames using **Union**. We will also learn how to clean up the inevitable duplicate records that occur when merging massive datasets.

### 🎯 Learning Objectives
* Understand the difference between combining data horizontally (Joins) and vertically (Unions).
* Learn the danger of positional stacking with `union()`.
* Master the safer, industry-standard approach using `unionByName()`.
* Remove exact duplicate rows using `distinct()`.
* Remove duplicate rows based on specific columns using `dropDuplicates()`.

In [ ]:
# 0. Setup: Let's start our SparkSession and create two DataFrames to stack.
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Union_and_Deduplication").master("local[*]").getOrCreate()

# Dataset 1: January Sales
jan_data = [
    (1, "Alice", 150.0),
    (2, "Bob", 200.0),
    (3, "Charlie", 50.0)
]
jan_df = spark.createDataFrame(jan_data, ["transaction_id", "customer_name", "amount"])

# Dataset 2: February Sales (Notice Bob's transaction is an accidental exact duplicate from Jan!)
feb_data = [
    (2, "Bob", 200.0),    # Duplicate!
    (4, "Diana", 300.0),
    (5, "Eve", 120.0)
]
feb_df = spark.createDataFrame(feb_data, ["transaction_id", "customer_name", "amount"])

print("--- January Sales ---")
jan_df.show()
print("--- February Sales ---")
feb_df.show()

## 1. Stacking Data: `union()` (The Dangerous Way)

The most basic way to stack two DataFrames is using the `.union()` method.

**How it works:** It stacks data based purely on **Column Position**. It takes column 1 from the bottom table and shoves it under column 1 of the top table, completely ignoring the column names!

If your two tables have the exact same columns in the exact same order, this works fine. But if the columns are out of order, you will corrupt your data.

<img src="../assets/Module_10/10_17_01.png" alt="A visual showing two blocks of data being stacked on top of each other. A warning sign indicates that if the blue 'Name' column is accidentally stacked under the green 'Age' column, the data becomes corrupted." width="75%">

In [ ]:
print("--- Standard union() ---")
# Our columns are perfectly aligned, so union() works as expected.
combined_df = jan_df.union(feb_df)
combined_df.show()

# Let's see what happens if the columns are out of order in the incoming data.
messy_feb_df = feb_df.select("amount", "transaction_id", "customer_name")

print("--- DANGER: union() with mismatched column order ---")
corrupted_df = jan_df.union(messy_feb_df)
corrupted_df.show()
print("Uh oh! Spark put the 'amount' (200.0) into the 'transaction_id' column because it was in the first position!")

## 2. Stacking Data: `unionByName()` (The Professional Way)

To avoid corrupting data, Data Engineers almost exclusively use **`unionByName()`**.

**How it works:** Instead of looking at the position of the column, it looks at the actual *name* of the column (e.g., "customer_name") and matches it to the same name in the other table, regardless of where it is located.

### 🌟 Pro-Tip: `allowMissingColumns=True`
What if February has a new column called `discount_code` that didn't exist in January? 
By default, `unionByName()` will crash if the schemas don't perfectly match. But if you pass the argument `allowMissingColumns=True`, Spark will gracefully stack the tables and fill the missing January spaces with `null`.

In [ ]:
print("--- unionByName() with mismatched columns ---")
# It automatically fixes the order and stacks them correctly!
safe_combined_df = jan_df.unionByName(messy_feb_df)
safe_combined_df.show()

print("--- unionByName() with missing columns ---")
# Let's add a new column to February that January doesn't have
import pyspark.sql.functions as F
advanced_feb_df = feb_df.withColumn("discount_code", F.lit("WINTER20"))

missing_cols_df = jan_df.unionByName(advanced_feb_df, allowMissingColumns=True)
missing_cols_df.show()
print("Spark safely filled the missing January discount codes with null!")

## 3. Deduplication: `distinct()` vs `dropDuplicates()`

Look closely at our stacked DataFrame. Because of a system glitch, Bob's transaction (ID 2) was recorded in both January and February. It is an exact duplicate.

When we combine massive files, duplicates are inevitable. We must remove them.

### 👯 `distinct()`
`distinct()` looks at the **entire row**. If every single column in Row A is exactly identical to every single column in Row B, it deletes one of them.

### 🎯 `dropDuplicates(['col_name'])`
Sometimes, you only want to look at a specific column. For example, if a user clicks a button 5 times, you might have 5 rows with the same `user_id` but slightly different timestamp columns. `distinct()` won't work because the timestamps are different. 
Instead, you use `dropDuplicates(['user_id'])` to tell Spark: *"If you see the same user_id more than once, keep the first one you see and throw the rest away."*

<img src="../assets/Module_10/10_17_02.png" alt="Diagram comparing distinct (comparing the whole row block) vs dropDuplicates (comparing only the ID column of the block)." width="75%">

In [ ]:
print("--- Data WITH Duplicates ---")
combined_df.show()

print("--- Using distinct() to remove exact matches ---")
# Removes Bob's exact duplicate row
dedup_distinct_df = combined_df.distinct()
dedup_distinct_df.show()

print("--- Using dropDuplicates() on a specific column ---")
# Let's add a fake row to demonstrate: Diana has a second transaction.
diana_row = spark.createDataFrame([(4, "Diana", 999.0)], ["transaction_id", "customer_name", "amount"])
df_with_diana = combined_df.union(diana_row)

# If we just want ONE record per customer_name, regardless of the amount:
dedup_specific_df = df_with_diana.dropDuplicates(["customer_name"])
dedup_specific_df.show()
print("Notice Diana only appears once, even though her amounts were different!")

## 🧑‍💻 The Modern Data Ecosystem: Role Comparison

How do different roles approach combining data?

| Feature | 🏛️ Traditional DBA | 🛠️ Data Engineer (You) | 🧠 Data Scientist |
| :--- | :--- | :--- | :--- |
| **Stacking Syntax** | `UNION ALL` in SQL. | **`df1.unionByName(df2, allowMissingColumns=True)` in PySpark.** | `pd.concat([df1, df2])` in Pandas. |
| **Deduplication** | `SELECT DISTINCT * FROM table` | **`df.dropDuplicates()` to sanitize massive Data Lake folders.** | Assumes the Data Engineer already removed duplicated web logs. |
| **Performance Focus** | Relies on the database engine. | **Knows that `distinct()` causes a massive Network Shuffle across the cluster to compare all rows, and uses it carefully.** | Memory limits on their local machine. |

---

### 📈 Career Progression Roadmap

1. **Junior DE:** Uses `union()` because it is faster to type. Experiences a catastrophic pipeline failure in production when the upstream API randomly changes the order of the columns in a JSON file.
2. **Mid-Level DE (Your Current Level):** Strictly uses `unionByName(allowMissingColumns=True)` to build bulletproof, flexible pipelines that don't crash when schemas evolve over time.
3. **Senior DE:** Optimizes the Deduplication process. A Senior DE knows that `distinct()` is a Wide Transformation (Shuffle). Before running `distinct()` on a Petabyte of data, they might repartition the data by a specific hash key to ensure the cluster doesn't choke during the shuffle.

### 🛠️ Where Data Engineering Fits in Modern Organizations
Companies generate data continuously (every day, every hour). The Data Engineer builds automated pipelines that wake up, grab today's new data file, `union` it with the historical historical master table, run `dropDuplicates` to ensure no double-counting occurred during a retry, and saves the clean result.

## 🔑 Key Takeaways

- **`union()`:** Stacks DataFrames vertically based on **column position**. Dangerous if schemas don't align perfectly.
- **`unionByName()`:** Stacks DataFrames safely based on **column names**.
- **`allowMissingColumns=True`:** A powerful argument for `unionByName` that fills non-matching columns with `null` instead of crashing.
- **`distinct()`:** Removes exact duplicate rows. It is a Wide Transformation (requires a Network Shuffle).
- **`dropDuplicates([cols])`:** Removes rows based on duplicated values in a specific subset of columns, keeping only the first occurrence.

## 📝 Knowledge Check Quiz

**Question 1:** You have a DataFrame of `2022_Sales` (Columns: ID, Amount, City) and `2023_Sales` (Columns: Amount, City, ID). What will happen if you use `.union()` to stack them?
A) Spark will automatically match the column names and stack them perfectly.
B) Spark will stack them based on position, putting the 'Amount' from 2023 under the 'ID' of 2022, corrupting the data.
C) Spark will throw an error and refuse to run.
D) Spark will delete the 2022 data.

**Question 2:** Which command should you use if you want to ensure that a DataFrame contains only ONE record per `user_id`, regardless of what data is in the other columns?
A) `df.distinct()`
B) `df.drop("user_id")`
C) `df.dropDuplicates(["user_id"])`
D) `df.filter("user_id == 1")`

**Question 3:** Why does calling `distinct()` on a massive DataFrame take a long time to execute in a Spark cluster?
A) Because it has to download the data to your laptop.
B) Because it deletes the SparkSession.
C) Because `distinct()` is a Wide Transformation. Spark must execute a Network Shuffle to compare rows across all the different Executor nodes to guarantee uniqueness.
D) Because it forces Spark to use Disk I/O like MapReduce.

---

*Answers: 1: B, 2: C, 3: C*

In [ ]:
# Clean up our session
spark.stop()
print("Spark session closed.")

### 🚀 What's Next?
Congratulations! You have completed **SECTION E: COMBINING DATA**.

You now know how to build complex, scalable Data Pipelines using purely PySpark Python code.

But what if you (or the analysts on your team) prefer writing traditional SQL? Spark has an incredible feature that allows you to execute raw SQL queries directly against your distributed DataFrames. In **SECTION F: ANALYTICAL PROCESSING**, we will dive into the **Spark SQL Engine**. See you in **10.18 Spark SQL**!